Preprocessing by Nguyen

In [215]:
import ast
import pandas as pd
import numpy as np
import spacy
from typing import Literal
import pickle

class Preprocess:
    def __init__(self):
        pass

Text = tuple[Literal[-1, 1], np.ndarray]
Dialog = list[Text]

def get_embeddings(file: str) -> dict:
    embeddings = {}
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], "float32")
            embeddings[word] = vector
    return embeddings

def preprocess_data(file: str = "Data/train.csv", embeddings_file: str = "vectors/glove.6B.50d.txt") -> list[Dialog]:
    embeddings = get_embeddings(embeddings_file)
    print("Embeddings loaded with size:", len(embeddings))
    nlp = spacy.load("en_core_web_sm")
    df = pd.read_csv(file)
    data: list[Dialog] = []
    for row in df.iloc:
        dialogue = ast.literal_eval(row['Dialogue'])['text']
        dialogue_data: Dialog = []
        for text in dialogue:
            if text['response'] == '$S$':
                continue
            if text['response'] == '$EXIT$':
                break
            role = 1 if text['role'] == 'A' else -1
            doc = nlp(text['response'])
            embeddings_data: list[np.ndarray] = []
            for token in doc:
                if token.lemma_ in embeddings:
                    embeddings_data.append(embeddings[token.lemma_])
            embeddings_data = np.array(embeddings_data)
            dialogue_data.append((role, embeddings_data))
        data.append(dialogue_data)
    with open("features.pkl", "wb") as f:
        pickle.dump(data, f)
    return data

def load_data(file: str = "features.pkl") -> list[Dialog]:
    with open(file, "rb") as f:
        data = pickle.load(f)
    return data

Preprocessing by Yibai

Notation:

- sequence == sentence
- batch == dialog
- inputs == X
- labels == y
- predictions == pred_y

In [216]:
from torch.nn.utils.rnn import PackedSequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.clip_grad import clip_grad_norm_
from torch.nn.utils.rnn import pad_sequence, pad_packed_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split

from tqdm import tqdm

In [217]:
def to_inputs_and_labels(data: list[Dialog]) -> tuple[list[list[np.ndarray]], list[list[Literal[-1, 1]]]]:
    '''Split data into inputs and labels

    Args:
        data(list[Dialog]): loaded training data, with non-uniform batch size
        and sequence length.  
                
    :Returns: tuple[inputs, labels] WHERE

        inputs(list[list[np.ndarray]]): input data, with non-uniform batch size
        and sequence length.
        
        labels(list[list[Literal[-1, 1]]]): labels for each sequence, with
        non-uniform batch size and sequence length.
    '''
    inputs = []
    labels = []
    for batch in data:
        inputs_batch = []
        labels_batch = []
        for pair in batch:
            inputs_batch.append(pair[1])
            labels_batch.append(pair[0])
        inputs.append(inputs_batch)
        labels.append(labels_batch)
    return (inputs, labels)

In [218]:
def is_valid_batch(
    inputs: list[np.ndarray],
    labels: list[Literal[-1, 1]],
    hidden_size: int
) -> bool:
    '''Verify whether the inputs and labels of a batch have valid vector representations.

    Args:
        inputs(list[np.ndarray]): list of all sequences in the batch
        labels(list[Literal[-1, 1]]): list of labels for each sequence in the batch
        hidden_size(int): hidden dimension size
    '''
    # Verify that inputs and labels are non-empty:
    if len(inputs) == 0 or len(labels) == 0:
        return False
    
    # Verify that all sequences in the batch are non-empty:
    for seq in inputs:
        if seq.shape == (0, ):
            return False
    
    # Verify that all tokens in the batch have the expected size of embedding:
    for seq in inputs:
        if seq.shape[1] != hidden_size:
            return False
    
    # Verify that inputs and labels have compatible dimensions:
    if len(inputs) != len(labels):
        return False

    return True

def filter_good_batches(
    inputs: list[list[np.ndarray]],
    labels: list[list[Literal[-1, 1]]],
    hidden_size: int
) -> tuple[list[list[np.ndarray]], list[list[Literal[-1, 1]]]]:
    '''Remove all batches containing invalid data

    Args:
        inputs(list[list[np.ndarray]]): inputs for all batches
        labels(list[list[Literal[-1, 1]]]): labels every sequences
    '''
    inputs_copy = inputs.copy()
    labels_copy = labels.copy()
    num_batch = len(inputs)
    assert(num_batch == len(labels))
    
    pop_count = 0
    for i in range(num_batch):
        if not is_valid_batch(inputs[i], labels[i], hidden_size):
            inputs_copy.pop(i - pop_count)
            labels_copy.pop(i - pop_count)
            pop_count += 1
    
    return (inputs_copy, labels_copy)

In [219]:
def collate_inputs(inputs: list[list[np.ndarray]]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    '''Pad sequences to global max_seq_len. Then pad batches to max_batch_size.
    Use this function instead of calling nn.utils.rnn.pad_sequence on each batch,
    because each batch has its own max_seq_len, and we need to use the global max.

    Args:
        inputs(list[list[np.ndarray]]): All input data
    
    Returns:
        tuple(tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]): Input tensor
        in the shape (num_batch, max_batch_size, max_seq_len),
        seq_lens tensor in the shape (num_batch, max_batch_size), and
        batch_sizes tensor in the shape (num_batch)
    '''
    # Flatten seqs across all dialogs and convert type from np.ndarray to torch.Tensor
    all_sequences = [torch.tensor(seq) for dialog in inputs for seq in dialog]

    max_seq_len = max(seq.size(0) for seq in all_sequences)
    padded_sequences = [F.pad(seq, (0, 0, 0, max_seq_len - seq.size(0))) for seq in all_sequences]

    # Group padded sequences back into dialog structure
    num_batch = len(inputs)
    max_batch_size = max(len(dialog) for dialog in inputs)

    padded_dialogs = []
    seq_lens = []
    batch_sizes = []

    idx = 0
    for dialog in inputs:
        padded_dialogs.append(torch.stack(padded_sequences[idx:idx + len(dialog)]))
        seq_lens.append(torch.tensor([len(seq) for seq in dialog]))
        batch_sizes.append(len(dialog))
        idx += len(dialog)

    # Pad dialogs and labels with zero sequences to match max_batch_size
    for i in range(num_batch):
        while len(padded_dialogs[i]) < max_batch_size:
            padded_dialogs[i] = torch.cat((padded_dialogs[i], torch.zeros(1, max_seq_len, padded_dialogs[i].size(-1))), dim=0)
            seq_lens[i] = torch.cat((seq_lens[i], torch.tensor([0])))

    return torch.stack(padded_dialogs), torch.stack(seq_lens), torch.tensor(batch_sizes)

In [ ]:
data = load_data("../features.pkl")
input_dim = 50

X, y = to_inputs_and_labels(data)
X, y = filter_good_batches(X, y, input_dim)
X, seq_lens, batch_sizes = collate_inputs(X)
y = pad_sequence([torch.tensor(batch) for batch in y]).T
y_mask = (y != 0)
y = (y + 1) / 2
y = y.long().float()

In [335]:
train_X, test_X, train_y, test_y, train_seq_lens, test_seq_lens, train_batch_sizes, test_batch_sizes = train_test_split(
    X, y, seq_lens, batch_sizes,
    test_size=0.2,      # 20% for testing
    random_state=42,    # Ensures reproducibility
    shuffle=True        # Shuffle dialogs for better generalization
)

In [363]:
from torch import Tensor
from torch.utils.data import Dataset

class LstmDataset(Dataset[tuple[Tensor, Tensor, Tensor, Tensor]]):
    def __init__(self, X, y, seq_lens, batch_sizes, len):
        self.X = X
        self.y = y
        self.seq_lens = seq_lens
        self.batch_sizes = batch_sizes
        self.len = len
       
    def __getitem__(self, index) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        return (X, y, seq_lens, batch_sizes)
    
    def __len__(self):
        return self.len

train_data = LstmDataset(train_X, train_y, train_seq_lens, train_batch_sizes, np.count_nonzero(y_mask))
test_data = LstmDataset(test_X, test_y, test_seq_lens, test_batch_sizes, np.count_nonzero(y_mask))

Simple LSTM Model

In [364]:
class TokenLSTM(nn.Module):
    '''Token LSTM Model

        This is a model which takes in token embeddings and generate a
        seq-aware token embedding for each sequence.

        X: input, shape: (num_batch, max_batch_size, max_seq_len, token_embed_size)
        
        pred_y: output, shape: (num_batch * max_batch_size, max_seq_len, hidden_size)
    '''
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(TokenLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True)
    
    def _pack_wrapper(self, X: torch.Tensor, seq_lens: torch.Tensor) -> PackedSequence:
        '''Mask the padding in input tensor to improve computational efficiency.
        This is a wrapper to specify reshaping operations involved.

        Returns:
            packed_X(torch.nn.utils.rnn.PackedSequence): packed input tensor
        '''
        flattened_X = X.view(-1, X.size(-2), X.size(-1))
        flattened_seq_lens = seq_lens.view(-1)

        # Filter out zero-length sequences, since pack_padded_sequence() only
        # accepts non-empty sequences.
        non_empty_mask = flattened_seq_lens > 0
        filtered_X = flattened_X[non_empty_mask]
        filtered_seq_lens = flattened_seq_lens[non_empty_mask]
        
        # Pack the sequences
        return pack_padded_sequence(filtered_X, 
                                    filtered_seq_lens, 
                                    batch_first=True,
                                    enforce_sorted=False)

    def _reintroduce_empty_seqs(self, pred_y, X, seq_lens):
        augmented_y = torch.zeros((X.size(0) * X.size(1), X.size(2), self.hidden_dim))
        non_empty_mask = seq_lens.view(-1) > 0
        augmented_y[non_empty_mask] = pred_y
        return augmented_y

    def forward(self, X: torch.Tensor, seq_lens: torch.Tensor):
        packed_X = self._pack_wrapper(X, seq_lens)
        packed_y, _ = self.lstm(packed_X)
        pred_y, _ = pad_packed_sequence(packed_y, batch_first=True)
        return self._reintroduce_empty_seqs(pred_y, X, seq_lens)

In [365]:
class SequenceLSTM(nn.Module):
    '''Sequence LSTM Model

        This is a model which takes in seq-aware token embeddings obtained from
        TokenLSTM and generate a dialog-aware seq embedding for each seq.

        X: input, shape: (num_batch * max_batch_size, max_seq_len, token_lstm_hidden_size)
        
        y: output, shape: (num_batch, max_batch_size, seq_embed_size)
    '''
    def __init__(self, input_dim, hidden_dim, num_layers):
        super(SequenceLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, X, batch_sizes):
        packed_X = pack_padded_sequence(X, batch_sizes, batch_first=True, enforce_sorted=False)
        packed_y, _ = self.lstm(packed_X)
        pred_y, _ = pad_packed_sequence(packed_y, batch_first=True)
        return pred_y

In [366]:
class BasicLSTM(nn.Module):
    def __init__(self, embed_size, h_1, h_2):
        super(BasicLSTM, self).__init__()
        self.token_lstm = TokenLSTM(embed_size, h_1, 1)
        self.sequence_lstm = SequenceLSTM(h_1, h_2, 1)
        self.fc = nn.Linear(h_2, 1)

    def forward(self, X, seq_lens, batch_sizes):
        X_1 = self.token_lstm(X, seq_lens)
        X_2 = self.sequence_lstm(X_1, batch_sizes)
        logits = self.fc(X_2)
        pred_y = torch.sigmoid(logits)
        return pred_y

In [367]:
hidden_dim = 128    # Hidden size for LSTM
device = 'cpu'

In [368]:
def train(
    model: BasicLSTM, 
    train_data: LstmDataset, 
    loss_function: nn.CrossEntropyLoss, 
    optimizer: torch.optim.Adam,
    seq_lens,
    batch_sizes,
    y_mask,
    num_epochs=5
) -> None:
    model.train()

    for epoch in tqdm(range(num_epochs)):
        total_loss = 0.0

        X, y, seq_lens, batch_sizes = train_data[0]
        X = X.to(device)
        y = y.to(device)

        y_pred = model.forward(X, seq_lens, batch_sizes).squeeze()

        loss = loss_function(y_pred[y_mask], y[y_mask])
        optimizer.zero_grad()
        loss.backward()

        # To avoid exploding gradients
        clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

        avg_loss = total_loss / len(train_data)
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [371]:
model = BasicLSTM(input_dim, hidden_dim, hidden_dim).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [381]:
train(model, train_data, loss_function, optimizer, seq_lens, batch_sizes, y_mask, num_epochs=20)

  5%|▌         | 1/20 [00:08<02:34,  8.12s/it]

Epoch 1/20, Loss: 4.2652


 10%|█         | 2/20 [00:15<02:13,  7.41s/it]

Epoch 2/20, Loss: 4.2636


 15%|█▌        | 3/20 [00:22<02:03,  7.25s/it]

...

Epoch 18/20, Loss: 4.2573


 95%|█████████▌| 19/20 [02:21<00:07,  7.27s/it]

Epoch 19/20, Loss: 4.2564


100%|██████████| 20/20 [02:27<00:00,  7.40s/it]

Epoch 20/20, Loss: 4.2559


In [376]:
for name, param in model.named_parameters():
    if param.grad is None:
        print(f"{name} has no gradient!")

In [379]:
torch.save(model, "lstm_basic.pt")

In [382]:
from sklearn.metrics import f1_score

def to_labels(y, threshold=0.5):
    labels = []
    for i in y:
        if i > threshold:
            labels.append(1.0)
        else:
            labels.append(0.0)
    return labels

X, y, seq_lens, batch_sizes = test_data[0]
y_pred = model.forward(X, seq_lens, batch_sizes).squeeze()
y_pred_int = to_labels(y_pred[y_mask].detach().numpy(), threshold=0.50)

f1_score(y[y_mask].detach().numpy(), y_pred_int)

0.7831808698443268